In [1]:
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from scipy.spatial.transform import Rotation
from scipy.spatial.distance import cdist
from Bio import pairwise2
from Bio.pairwise2 import format_alignment
import warnings
warnings.filterwarnings('ignore')

class TBM_RNA_Folder:
    
    def __init__(self):
        self.template_db = self._load_template_database()
        self.gap_open = -10
        self.gap_extend = -0.5
        self.c1_distance = 5.9
        
    def _load_template_database(self):
        templates = []
        pdb_dir = "/kaggle/input/stanford-rna-3d-folding-2/PDB_RNA"
        
        for filename in os.listdir(pdb_dir):
            if filename.endswith(".cif"):
                try:
                    structure = self._parse_cif(os.path.join(pdb_dir, filename))
                    if structure and len(structure['sequence']) >= 10:
                        templates.append(structure)
                except:
                    continue
        
        return templates
    
    def _parse_cif(self, filepath):
        nucleotides = {}
        seq = []
        coords = []
        
        with open(filepath, 'r') as f:
            lines = f.readlines()
        
        current_res = None
        current_coords = {}
        
        for line in lines:
            if line.startswith('_atom_site.'):
                continue
                
            parts = line.strip().split()
            if len(parts) < 8:
                continue
            
            atom_name = parts[3]
            res_name = parts[5]
            res_id = parts[6]
            x, y, z = map(float, parts[10:13])
            
            if current_res != res_id:
                if current_res is not None:
                    if 'C1\'' in current_coords:
                        seq.append(self._get_nucleotide_code(res_name))
                        coords.append(current_coords['C1\''])
                current_res = res_id
                current_coords = {}
            
            if atom_name in ["C1'", "C2'", "C3'", "C4'", "C5'", "O3'", "O5'", "P"]:
                current_coords[atom_name] = np.array([x, y, z])
        
        if current_res is not None and 'C1\'' in current_coords:
            seq.append(self._get_nucleotide_code(res_name))
            coords.append(current_coords['C1\''])
        
        return {
            'sequence': ''.join(seq),
            'coords': np.array(coords),
            'id': os.path.basename(filepath).replace('.cif', '')
        }
    
    def _get_nucleotide_code(self, res_name):
        code_map = {
            'A': 'A', 'ADE': 'A', 'DA': 'A',
            'U': 'U', 'URA': 'U', 'DT': 'U',
            'G': 'G', 'GUA': 'G', 'DG': 'G',
            'C': 'C', 'CYT': 'C', 'DC': 'C'
        }
        return code_map.get(res_name[:1], 'N')
    
    def search_templates(self, query_seq, top_k=20):
        scores = []
        
        for template in self.template_db:
            alignments = pairwise2.align.globalms(
                query_seq, template['sequence'],
                2, -1, self.gap_open, self.gap_extend
            )
            
            if alignments:
                best_aln = alignments[0]
                identity = sum(a == b for a, b in zip(best_aln.seqA, best_aln.seqB) if a != '-' and b != '-')
                length = sum(1 for a, b in zip(best_aln.seqA, best_aln.seqB) if a != '-' or b != '-')
                
                if length > 0:
                    similarity = identity / length
                    coverage = sum(1 for a in best_aln.seqA if a != '-') / len(query_seq)
                    score = similarity * coverage
                    
                    scores.append({
                        'template': template,
                        'score': score,
                        'alignment': best_aln,
                        'similarity': similarity,
                        'coverage': coverage
                    })
        
        scores.sort(key=lambda x: x['score'], reverse=True)
        return scores[:top_k]
    
    def transfer_coordinates(self, query_seq, template, alignment):
        query_aligned = alignment.seqA
        template_aligned = alignment.seqB
        
        template_coords = template['coords']
        transferred = []
        
        q_idx, t_idx = 0, 0
        
        for q_base, t_base in zip(query_aligned, template_aligned):
            if q_base != '-' and t_base != '-':
                if t_idx < len(template_coords):
                    transferred.append(template_coords[t_idx].copy())
                else:
                    transferred.append(np.zeros(3))
                q_idx += 1
                t_idx += 1
            elif q_base != '-' and t_base == '-':
                transferred.append(None)
                q_idx += 1
            elif q_base == '-' and t_base != '-':
                t_idx += 1
        
        return transferred
    
    def fill_gaps(self, coords_list, query_seq, confidence):
        filled_coords = []
        
        for i, coords in enumerate(coords_list):
            if coords is not None:
                filled_coords.append(coords)
            else:
                if i == 0 or i == len(coords_list) - 1:
                    filled_coords.append(self._extend_terminus(filled_coords, i))
                else:
                    prev = filled_coords[-1]
                    next_valid = self._find_next_valid(coords_list, i)
                    
                    if next_valid is not None:
                        next_coords = coords_list[next_valid]
                        filled_coords.append(self._interpolate_gap(prev, next_coords, next_valid - i))
                    else:
                        filled_coords.append(self._extend_terminus(filled_coords, i))
        
        coords_array = np.array(filled_coords)
        
        constraint_strength = 0.8 * (1 - min(confidence, 0.8))
        coords_array = self._adaptive_refinement(coords_array, query_seq, constraint_strength)
        
        return coords_array
    
    def _extend_terminus(self, existing_coords, position):
        if not existing_coords:
            return np.random.normal(0, 1, 3)
        
        if position == 0:
            direction = existing_coords[0] - np.mean(existing_coords[:3], axis=0)
        else:
            direction = existing_coords[-1] - existing_coords[-2]
        
        if np.linalg.norm(direction) < 0.001:
            direction = np.random.randn(3)
        
        direction = direction / np.linalg.norm(direction) * self.c1_distance
        return existing_coords[-1] + direction
    
    def _find_next_valid(self, coords_list, start_idx):
        for i in range(start_idx + 1, len(coords_list)):
            if coords_list[i] is not None:
                return i
        return None
    
    def _interpolate_gap(self, start_coords, end_coords, gap_length):
        if gap_length <= 1:
            return (start_coords + end_coords) / 2
        
        vec = end_coords - start_coords
        distance = np.linalg.norm(vec)
        target_distance = gap_length * self.c1_distance
        
        if distance < target_distance * 0.7:
            base_step = vec / gap_length
            coords = start_coords + base_step
            
            amplitude = min(3.0, distance * 0.3)
            for j in range(1, gap_length - 1):
                perturbation = np.random.randn(3)
                perturbation = perturbation / np.linalg.norm(perturbation) * amplitude
                coords = np.vstack([coords, start_coords + base_step * (j + 1) + perturbation])
            
            return coords[-1]
        else:
            t = 1 / (gap_length + 1)
            return start_coords + vec * t
    
    def _adaptive_refinement(self, coords, sequence, strength):
        if strength < 0.1:
            return coords
        
        coords_tensor = torch.tensor(coords, dtype=torch.float64, requires_grad=True)
        
        optimizer = optim.LBFGS([coords_tensor], lr=0.1, max_iter=100)
        
        def closure():
            optimizer.zero_grad()
            loss = 0.0
            
            distances = torch.cdist(coords_tensor, coords_tensor)
            
            for i in range(len(coords_tensor) - 1):
                dist = torch.norm(coords_tensor[i+1] - coords_tensor[i])
                target = self.c1_distance
                loss += torch.abs(dist - target) * strength
            
            mask = (distances < 3.0) & (distances > 0)
            loss += torch.sum(1.0 / (distances[mask] + 1e-6)) * strength * 0.1
            
            for i in range(len(sequence) - 4):
                pairs = [('G', 'C'), ('C', 'G'), ('A', 'U'), ('U', 'A')]
                if (sequence[i], sequence[i+4]) in pairs:
                    dist = torch.norm(coords_tensor[i] - coords_tensor[i+4])
                    loss += torch.abs(dist - 10.5) * strength * 0.5
            
            loss.backward()
            return loss
        
        optimizer.step(closure)
        
        return coords_tensor.detach().numpy()

class EnhancedDRfold2Selector:
    
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    def score_models_double(self, models):
        scores = []
        
        for model in models:
            model_tensor = torch.tensor(model, dtype=torch.float64).to(self.device)
            
            distances = torch.cdist(model_tensor, model_tensor)
            
            contact_score = self._calculate_contact_score(distances)
            compactness_score = self._calculate_compactness(model_tensor)
            clash_score = self._calculate_clash_score(distances)
            
            total_score = 0.5 * contact_score + 0.3 * compactness_score + 0.2 * clash_score
            scores.append(float(total_score.cpu()))
        
        return scores
    
    def _calculate_contact_score(self, distances):
        mask = (distances > 4.0) & (distances < 8.0)
        return torch.sum(mask.float()) / (distances.shape[0] ** 2)
    
    def _calculate_compactness(self, coords):
        centroid = torch.mean(coords, dim=0)
        distances = torch.norm(coords - centroid, dim=1)
        return 1.0 / (torch.std(distances) + 1e-6)
    
    def _calculate_clash_score(self, distances):
        mask = distances < 3.0
        clash_count = torch.sum(mask.float()) - distances.shape[0]
        return 1.0 / (clash_count + 1.0)

class TBM_DL_Hybrid:
    
    def __init__(self):
        self.tbm_folder = TBM_RNA_Folder()
        self.drfold2_selector = EnhancedDRfold2Selector()
        self.use_tbm_for_short = True
        self.min_length_for_dl = 50
    
    def predict_structure(self, sequence, max_models=5):
        if len(sequence) < self.min_length_for_dl and self.use_tbm_for_short:
            return self._tbm_prediction(sequence, max_models)
        else:
            try:
                return self._hybrid_prediction(sequence, max_models)
            except:
                return self._tbm_prediction(sequence, max_models)
    
    def _tbm_prediction(self, sequence, max_models):
        templates = self.tbm_folder.search_templates(sequence, top_k=10)
        models = []
        
        for template_info in templates[:3]:
            transferred = self.tbm_folder.transfer_coordinates(
                sequence, 
                template_info['template'], 
                template_info['alignment']
            )
            
            model = self.tbm_folder.fill_gaps(
                transferred, 
                sequence, 
                template_info['similarity']
            )
            models.append(model)
        
        while len(models) < max_models:
            if templates:
                template = templates[np.random.randint(0, min(3, len(templates)))]
                transferred = self.tbm_folder.transfer_coordinates(
                    sequence, template['template'], template['alignment']
                )
                model = self.tbm_folder.fill_gaps(transferred, sequence, template['similarity'] * 0.8)
            else:
                model = self._generate_de_novo(sequence)
            models.append(model)
        
        return models[:max_models]
    
    def _hybrid_prediction(self, sequence, max_models):
        tbm_models = self._tbm_prediction(sequence, 2)
        
        try:
            dl_models = self._run_drfold2(sequence, 3)
            models = tbm_models + dl_models
        except:
            models = tbm_models + self._generate_de_novo_models(sequence, 3)
        
        if len(models) > max_models:
            scores = self.drfold2_selector.score_models_double(models)
            sorted_indices = np.argsort(scores)[::-1]
            models = [models[i] for i in sorted_indices[:max_models]]
        
        return models[:max_models]
    
    def _run_drfold2(self, sequence, num_models):
        models = []
        
        for _ in range(num_models):
            model = self._generate_de_novo(sequence)
            
            model_tensor = torch.tensor(model, dtype=torch.float64, requires_grad=True).to(
                self.drfold2_selector.device
            )
            
            optimizer = optim.LBFGS([model_tensor], lr=0.01, max_iter=50)
            
            def closure():
                optimizer.zero_grad()
                loss = 0.0
                
                distances = torch.cdist(model_tensor, model_tensor)
                
                for i in range(len(model_tensor) - 1):
                    dist = torch.norm(model_tensor[i+1] - model_tensor[i])
                    loss += torch.abs(dist - 5.9)
                
                mask = distances < 3.0
                loss += torch.sum(3.0 - distances[mask]) * 0.1
                
                loss.backward()
                return loss
            
            for _ in range(3):
                optimizer.step(closure)
            
            models.append(model_tensor.cpu().detach().numpy())
        
        return models
    
    def _generate_de_novo(self, sequence):
        length = len(sequence)
        twist = np.deg2rad(32.7)
        radius = 9.0
        
        coords = []
        for i in range(length):
            angle = i * twist * (0.95 + 0.1*np.random.randn())
            x = radius * np.cos(angle) * (0.9 + 0.2*np.random.rand())
            y = radius * np.sin(angle) * (0.9 + 0.2*np.random.rand())
            z = i * 2.8 * (0.95 + 0.1*np.random.randn())
            coords.append([x, y, z])
        
        coords = np.array(coords)
        
        for i in range(length - 4):
            pairs = [('G', 'C'), ('C', 'G'), ('A', 'U'), ('U', 'A')]
            if (sequence[i], sequence[i+4]) in pairs:
                vec = coords[i+4] - coords[i]
                dist = np.linalg.norm(vec)
                if dist > 0:
                    scale = 10.5 / dist
                    mid = (coords[i] + coords[i+4]) / 2
                    coords[i] = mid + (coords[i] - mid) * scale * 0.8
                    coords[i+4] = mid + (coords[i+4] - mid) * scale * 0.8
        
        return coords
    
    def _generate_de_novo_models(self, sequence, num_models):
        return [self._generate_de_novo(sequence) for _ in range(num_models)]

def create_hybrid_submission(test_file, output_file='submission.csv'):
    predictor = TBM_DL_Hybrid()
    test_df = pd.read_csv(test_file)
    
    all_data = []
    
    for idx, row in test_df.iterrows():
        target_id = row['target_id']
        sequence = row['sequence']
        
        print(f"Processing {target_id} ({len(sequence)} nt)...")
        
        models = predictor.predict_structure(sequence, 5)
        
        for i, nucleotide in enumerate(sequence):
            residue_id = f"{target_id}_{i+1}"
            row_data = {
                'ID': residue_id,
                'resname': nucleotide,
                'resid': i + 1
            }
            
            for model_idx in range(1, 6):
                if model_idx - 1 < len(models) and i < len(models[model_idx-1]):
                    coords = models[model_idx-1][i]
                    row_data[f'x_{model_idx}'] = float(coords[0])
                    row_data[f'y_{model_idx}'] = float(coords[1])
                    row_data[f'z_{model_idx}'] = float(coords[2])
                else:
                    row_data[f'x_{model_idx}'] = 0.0
                    row_data[f'y_{model_idx}'] = 0.0
                    row_data[f'z_{model_idx}'] = 0.0
            
            all_data.append(row_data)
    
    submission_df = pd.DataFrame(all_data)
    
    column_order = ['ID', 'resname', 'resid']
    for i in range(1, 6):
        column_order.extend([f'x_{i}', f'y_{i}', f'z_{i}'])
    
    submission_df = submission_df[column_order]
    
    for i in range(1, 6):
        for coord in ['x', 'y', 'z']:
            col = f'{coord}_{i}'
            submission_df[col] = pd.to_numeric(submission_df[col], errors='coerce')
            submission_df[col] = submission_df[col].fillna(0.0)
    
    submission_df.to_csv(output_file, index=False, float_format='%.3f')
    
    print(f"Submission saved to {output_file}")
    print(f"Shape: {submission_df.shape}")
    
    return submission_df

if __name__ == "__main__":
    import os
    
    submission = create_hybrid_submission(
        '/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv',
        'submission.csv'
    )

/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


Processing 8ZNQ (30 nt)...
Processing 9IWF (69 nt)...
Processing 9JGM (210 nt)...
Processing 9MME (4640 nt)...
Processing 9J09 (214 nt)...
Processing 9E9Q (101 nt)...
Processing 9CFN (59 nt)...
Processing 9OBM (73 nt)...
Processing 9G4P (68 nt)...
Processing 9G4Q (104 nt)...
Processing 9G4R (47 nt)...
Processing 9RVP (34 nt)...
Processing 9JFS (246 nt)...
Processing 9LEC (378 nt)...
Processing 9LEL (476 nt)...
Processing 9I9W (28 nt)...
Processing 9HRO (35 nt)...
Processing 9QZJ (19 nt)...
Processing 9JFO (195 nt)...
Processing 9OD4 (23 nt)...
Processing 9WHV (80 nt)...
Processing 9E74 (255 nt)...
Processing 9E75 (165 nt)...
Processing 9G4J (334 nt)...
Processing 9KGG (267 nt)...
Processing 9EBP (81 nt)...
Processing 9LJN (71 nt)...
Processing 9ZCC (1460 nt)...
Submission saved to submission.csv
Shape: (9762, 18)
